In [1]:
# -*- coding: utf-8 -*-
"""
CTR — DCN-V2 + (DIEN or BST) + DUSIN Segment (Extractor/Activation) + Aux Losses
- 시퀀스: 해시 임베딩, Lazy 파싱
- 본체: CrossNetMix(DCN-v2) + Deep
- DUSIN:
  * Segment Extractor: segment_cols로 만든 명시 세그먼트별 프로토타입(은닉 last의 EMA) + 세그 분류 보조손실
  * Segment Activation: vU(A)=vA⊙S_i, vU(S)=Attn(H, query=S_i) → 최종 concat
- DIEN Aux Loss: 다음 행동 예측 (in-batch + sampled negatives)
- 안정성: AMP, NaN-safe, fp16-safe masking
"""
import os, re, json, random, gc, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# ================= Utils =================
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]; toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s); toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: pass
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

# ================= Config =================
class Args:
    # files / columns
    train="./train.parquet"; test="./test.parquet"
    label="clicked"; seq_col="seq"; id_col="ID"

    # features
    sample_subset=1.0
    max_seq_len=50
    HASH_BUCKETS=262_144
    seq_emb_dim=64
    dropout=0.2

    # sequence backbone: "dien" or "bst"
    seq_backbone="dien"
    # BST hyperparams
    bst_layers=2; bst_nhead=4; bst_ffn=128

    # DCN-V2
    cross_layers=3; cross_low_rank=32; cross_num_experts=4
    deep_units=[512,256,128]

    # segment settings
    segment_cols=["gender","age_group"]
    seg_ema=0.9           # EMA momentum for segment prototype
    use_seg_aux=True      # 세그먼트 분류 보조손실 on/off
    seg_aux_lambda=0.1    # 세그먼트 보조손실 가중치

    # DIEN next-action aux loss
    use_next_aux=True
    next_aux_lambda=0.2   # 다음행동 보조손실 가중치
    next_neg_k=32         # 샘플 네거티브 수 (in-batch+uniform 혼합)

    # train
    epochs=3; batch_size=512; lr=1e-3; patience=2
    seed=42; device="auto"; num_workers=0; pin_memory=False

args=Args()
set_seed(args.seed); device=infer_device(args.device)

# ================= (옵션) 더미 데이터 생성 =================
def create_dummy(num_rows, is_test=False):
    rng=np.random.default_rng(0)
    df=pd.DataFrame({
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0,7,num_rows),
        "hour": rng.integers(0,24,num_rows),
        "l_feat_14": rng.integers(1000,1050, num_rows),
    })
    seqs=[]
    for _ in range(num_rows):
        fmt=rng.choice(["pipe","comma","json","empty","single"])
        L=int(rng.integers(1,args.max_seq_len+10))
        items=[str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe": seqs.append("|".join(items))
        elif fmt=="comma": seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty": seqs.append(np.nan)
        else: seqs.append(items[0])
    df[args.seq_col]=seqs
    if is_test: df[args.id_col]=np.arange(num_rows)
    else: df[args.label]=rng.integers(0,2,num_rows)
    return df

if not os.path.exists(args.train):
    print("train.parquet 없음 → 더미 20k 생성"); create_dummy(20_000,False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 더미 5k 생성"); create_dummy(5_000,True).to_parquet(args.test)

# ================= Load & Plan =================
print("[1/9] Load parquet ...")
train=pd.read_parquet(args.train); test=pd.read_parquet(args.test)
assert args.label in train.columns and args.seq_col in train.columns

if 0<args.sample_subset<1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train=train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

must_drop={args.label,args.seq_col,args.id_col}
base_cols=[c for c in train.columns if c not in must_drop]

fixed_cats = [c for c in ["gender","age_group","inventory_id","day_of_week","hour"] if c in base_cols]
all_lfeats = [c for c in base_cols if c.startswith("l_feat_")]

preferred_targets = ["ad_id","creative_id","l_feat_14"]
present_targets = [c for c in preferred_targets if c in base_cols]
target_name = present_targets[0] if len(present_targets)>0 else ( "l_feat_14" if "l_feat_14" in all_lfeats else None )

CAT_CARD_MAX = 200_000
cand_cats = fixed_cats.copy()
tr_tmp, va_tmp = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
for c in all_lfeats:
    nunq = max(tr_tmp[c].nunique(dropna=True), va_tmp[c].nunique(dropna=True))
    if nunq <= CAT_CARD_MAX: cand_cats.append(c)
if target_name is not None and target_name not in cand_cats and target_name in base_cols:
    cand_cats.append(target_name)
cont_cols=[c for c in base_cols if c not in cand_cats]
print(f"[1.1] target_name={target_name}  cont={len(cont_cols)}  cat={len(cand_cats)}")

# segment id space (from train split estimate)
def build_segment_key(df, cols):
    if len(cols)==0: return pd.Series(["__ALL__"]*len(df))
    return df[cols].astype(str).agg("|".join, axis=1)
seg_key_tr = build_segment_key(tr_tmp, args.segment_cols)
seg_key_va = build_segment_key(va_tmp, args.segment_cols)
seg_uniqs = pd.Index(pd.concat([seg_key_tr, seg_key_va]).unique())
SEG_OOV = len(seg_uniqs)
seg2idx = {v:i for i,v in enumerate(seg_uniqs)}
del tr_tmp, va_tmp; gc.collect()

# final split
tr,va=train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label]); del train; gc.collect()

# categorical maps/cards
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats=pd.Categorical(tr[c])
    cat_maps[c]={v:i for i,v in enumerate(cats.categories)}
    cat_cards[c]=len(cats.categories)

# seq hashing
PAD_ID, OOV_ID, SEQ_BASE = 0, 1, 2
seq_vocab_size = args.HASH_BUCKETS + SEQ_BASE
def seq_to_ids_hash(lst, max_len=50):
    ids=[SEQ_BASE + (int(t)%args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids)<max_len: ids=[PAD_ID]*(max_len-len(ids))+ids
    return np.array(ids, dtype=np.int32)

# ================= Dataset =================
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, seg_cols, label_col=None):
        self.df=df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len=max_seq_len; self.has_label=label_col is not None; self.label_col=label_col
        self.seg_cols=seg_cols

        self.Xc=self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats={c:self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values for c in self.cat_cols}
        if self.has_label: self.y=self.df[self.label_col].astype(np.float32).values

        key = build_segment_key(self.df, self.seg_cols)
        self.seg_ids = key.map(seg2idx).fillna(SEG_OOV).astype(np.int64).values

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        xc=torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats={c: torch.tensor(self.Xcats[c][idx],dtype=torch.long) for c in self.cat_cols}
        lst=parse_seq_string(self.df.at[idx,self.seq_col])
        seq=torch.from_numpy(seq_to_ids_hash(lst, self.max_seq_len)).long()
        seg_id = torch.tensor(self.seg_ids[idx], dtype=torch.long)
        if self.has_label: return xc, cats, seq, seg_id, torch.tensor(self.y[idx],dtype=torch.float32)
        return xc, cats, seq, seg_id

# ================= DCN-V2 =================
class CrossLayerMix(nn.Module):
    def __init__(self, dim, low_rank=32, num_experts=4):
        super().__init__()
        self.U=nn.Parameter(torch.randn(num_experts, dim, low_rank)*0.01)
        self.V=nn.Parameter(torch.randn(num_experts, low_rank, dim)*0.01)
        self.C=nn.Parameter(torch.zeros(num_experts, dim))
        self.gating=nn.Linear(dim, num_experts, bias=False)
        self.bias=nn.Parameter(torch.zeros(dim))
        self.num_experts=num_experts
    def forward(self, x0, x):
        gate=torch.softmax(self.gating(x), dim=-1)            # (B,E)
        Xu=torch.einsum("bd,edr->ber", x, self.U)             # (B,E,r)
        Xuv=torch.einsum("ber,erd->bed", Xu, self.V)+self.C   # (B,E,d)
        mix=torch.einsum("be,bed->bd", gate, Xuv)             # (B,d)
        return x0*mix + x + self.bias

class CrossNetMix(nn.Module):
    def __init__(self, dim, num_layers=3, low_rank=32, num_experts=4):
        super().__init__()
        self.layers=nn.ModuleList([CrossLayerMix(dim, low_rank, num_experts) for _ in range(num_layers)])
    def forward(self, x):
        x0=x; xl=x
        for layer in self.layers: xl=layer(x0, xl)
        return xl

# ================= Blocks =================
class DINActivationUnit(nn.Module):
    def __init__(self, dim_q, dim_k, hidden=[64,32], dropout=0.0):
        super().__init__()
        in_dim = dim_q + dim_k + dim_q + dim_k
        layers=[]
        for h in hidden:
            layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        layers += [nn.Linear(in_dim,1)]
        self.net=nn.Sequential(*layers)
    def forward(self, q, K):  # q:(B,D), K:(B,L,D)
        B,L,Dk=K.shape
        q_exp=q.unsqueeze(1).expand(-1,L,-1)
        z=torch.cat([q_exp, K, q_exp-K, q_exp*K], dim=2)
        return self.net(z).squeeze(2)  # (B,L)

class DIENBackbone(nn.Module):
    def __init__(self, vocab_size, emb_dim, padding_idx=0, dropout=0.2):
        super().__init__()
        self.emb=nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.gru=nn.GRU(input_size=emb_dim, hidden_size=emb_dim, batch_first=True)
        self.drop=nn.Dropout(dropout)
    def forward(self, seq_ids):
        K0=self.emb(seq_ids)
        H,_=self.gru(K0)
        return self.drop(H)   # (B,L,D)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe=torch.zeros(max_len, d_model)
        pos=torch.arange(0,max_len).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2)*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):  # (B,L,D)
        return x + self.pe[:,:x.size(1),:]

class BSTBackbone(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead=4, num_layers=2, dim_ff=128, dropout=0.2, padding_idx=0):
        super().__init__()
        self.emb=nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.pos=PositionalEncoding(emb_dim, max_len=2048)
        enc_layer=nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead, dim_feedforward=dim_ff,
                                             batch_first=True, dropout=dropout, activation="relu", norm_first=True)
        self.enc=nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.drop=nn.Dropout(dropout)
    def forward(self, seq_ids):
        K0=self.emb(seq_ids); K0=self.pos(K0)
        mask=(seq_ids==PAD_ID)
        H=self.enc(K0, src_key_padding_mask=mask)
        return self.drop(H)  # (B,L,D)

# ================= Segment Extractor (EMA Bank + Aux CE) =================
class SegmentPrototypeBank(nn.Module):
    def __init__(self, num_segments, dim, oov_index, momentum=0.9):
        super().__init__()
        self.register_buffer("bank", torch.zeros(num_segments+1, dim))
        self.momentum = float(momentum)
        self.oov=int(oov_index)
    @torch.no_grad()
    def ema_update(self, seg_ids, v_last):
        # v_last:(B,D), seg_ids:(B,)
        if v_last.dtype!=self.bank.dtype: v_last=v_last.to(self.bank.dtype)
        for s in torch.unique(seg_ids):
            mask=(seg_ids==s).unsqueeze(1)  # (B,1)
            if mask.any():
                v_seg = v_last.masked_select(mask).view(-1, v_last.size(1)).mean(dim=0)
                v_seg = torch.nan_to_num(v_seg, nan=0.0, posinf=0.0, neginf=0.0)
                self.bank[s] = self.momentum*self.bank[s] + (1.0-self.momentum)*v_seg
    def lookup(self, seg_ids):   # (B,D)
        return self.bank[seg_ids]

class SegmentClassifier(nn.Module):
    """보조손실: 개인 관심 v_last → 세그먼트 분류"""
    def __init__(self, dim, num_segments_plus_oov):
        super().__init__()
        self.head=nn.Linear(dim, num_segments_plus_oov)
    def forward(self, v_last):
        return self.head(v_last)  # logits

# ================= Model =================
class DCN_DUSIN_Full(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, num_segments, seg_oov_index,
                 target_name=None, seq_emb_dim=64, seq_backbone="dien",
                 bst_cfg=None, deep_units=[512,256,128],
                 cross_layers=3, cross_low_rank=32, cross_num_experts=4, dropout=0.2,
                 seg_momentum=0.9, use_seg_aux=True):
        super().__init__()
        self.has_cont=cont_dim>0
        if self.has_cont: self.bn=nn.BatchNorm1d(cont_dim)

        # categorical
        self.cat_embs=nn.ModuleDict()
        for name,card in cat_cards.items():
            dim=emb_dim_from_card(card+1)
            self.cat_embs[name]=nn.Embedding(card+1+1, dim)
        self.cat_total_dim=sum(emb.embedding_dim for emb in self.cat_embs.values())
        self.tab_dim=(cont_dim if self.has_cont else 0)+self.cat_total_dim

        # DCN-V2 + Deep
        self.cross=CrossNetMix(self.tab_dim, num_layers=cross_layers,
                               low_rank=cross_low_rank, num_experts=cross_num_experts)
        in_dim=self.tab_dim; deep_layers=[]
        for h in deep_units:
            deep_layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]; in_dim=h
        self.deep=nn.Sequential(*deep_layers)

        # target emb
        self.target_name=target_name if (target_name in self.cat_embs) else None
        self.tproj=None
        if self.target_name is not None:
            tdim=self.cat_embs[self.target_name].embedding_dim
            if tdim!=seq_emb_dim: self.tproj=nn.Linear(tdim, seq_emb_dim, bias=False)

        # sequence backbone
        self.seq_backbone=seq_backbone
        if seq_backbone=="bst":
            cfg = bst_cfg or {"nhead":4,"num_layers":2,"dim_ff":128}
            self.seq_enc=BSTBackbone(seq_vocab_size, seq_emb_dim, nhead=cfg["nhead"],
                                     num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"], dropout=dropout, padding_idx=PAD_ID)
        else:
            self.seq_enc=DIENBackbone(seq_vocab_size, seq_emb_dim, padding_idx=PAD_ID, dropout=dropout)

        # segment prototype bank + aux classifier
        self.seg_bank = SegmentPrototypeBank(num_segments, seq_emb_dim, seg_oov_index, momentum=seg_momentum)
        self.use_seg_aux = use_seg_aux
        self.seg_cls = SegmentClassifier(seq_emb_dim, num_segments+1) if use_seg_aux else None

        # DUSIN activation
        self.act_seg = DINActivationUnit(dim_q=seq_emb_dim, dim_k=seq_emb_dim, hidden=[64,32], dropout=dropout)

        # next-action projection (for aux loss): use item embedding table from backbone if DIEN else new table
        # 여기서는 backbone의 임베딩 공유 (해시 vocab)
        if isinstance(self.seq_enc, DIENBackbone) or isinstance(self.seq_enc, BSTBackbone):
            self.item_emb = self.seq_enc.emb     # share weights

        # final head: [x_cross, x_deep, v_last, vU_A, vU_S, S_i]
        final_in=self.tab_dim + deep_units[-1] + seq_emb_dim*4
        self.head=nn.Sequential(nn.Linear(final_in,256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256,1))

    def forward(self, xc, xcats, seq_ids, seg_ids):
        # Tabular
        outs=[]
        if self.has_cont: outs.append(self.bn(xc))
        if len(self.cat_embs)>0: outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))
        x0 = torch.cat(outs, dim=1) if len(outs)>1 else outs[0]
        x_cross=self.cross(x0); x_deep=self.deep(x0)

        # Sequence -> H, v_last(근사)
        H=self.seq_enc(seq_ids)     # (B,L,D)
        L=H.size(1)
        # 마지막 non-PAD 위치 근사: 시간가중 max
        t = torch.arange(L, device=H.device).float().view(1,L,1) + 1.0
        mask=(seq_ids!=PAD_ID).float().unsqueeze(2)  # (B,L,1)
        Hwm=H * t * mask
        v_last = Hwm.max(dim=1).values
        v_last = torch.where(mask.sum(dim=1)>0, v_last, torch.zeros_like(v_last))

        # Segment prototype (EMA update & lookup)
        with torch.no_grad():
            self.seg_bank.ema_update(seg_ids, v_last.detach())
        S_i = self.seg_bank.lookup(seg_ids)  # (B,D)

        # target emb
        if self.target_name is not None:
            vA=self.cat_embs[self.target_name](xcats[self.target_name])
            if self.tproj is not None: vA=self.tproj(vA)
        else:
            vA=torch.zeros_like(S_i)

        # Activation
        vU_A = vA * S_i
        logits_w = self.act_seg(S_i, H)  # (B,L)
        neg = torch.finfo(H.dtype).min
        logits_w = logits_w.masked_fill((seq_ids==PAD_ID), neg)
        valid = (seq_ids!=PAD_ID).any(dim=1, keepdim=True)
        maxv = torch.where(valid, logits_w.max(dim=1, keepdim=True).values, torch.zeros_like(logits_w[:,:1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        vU_S = torch.sum(H * alpha.unsqueeze(2), dim=1)

        # Head
        z=torch.cat([x_cross, x_deep, v_last, vU_A, vU_S, S_i], dim=1)
        logits=self.head(z).squeeze(1)

        # Aux logits (optional): segment classification
        seg_logits = self.seg_cls(v_last) if self.seg_cls is not None else None

        # For next-action aux: return H and seq_ids to compute outside
        return logits, seg_logits, H

# ================= Next-action Aux Loss (NCE) =================
def next_action_aux_loss(H, seq_ids, item_emb, neg_k=32):
    """
    H:(B,L,D), seq_ids:(B,L) -> 다음 행동 예측 보조손실 (NCE/InfoNCE 스타일)
    - 연산은 전부 float32로 수행 (AMP/fp16 안전)
    """
    B, L, D = H.shape
    if L <= 1:
        return H.new_zeros((), dtype=torch.float32)

    # 1) 타깃 시프트 (t -> t+1)
    target = seq_ids[:, 1:].contiguous()       # (B, L-1)
    Hctx   = H[:, :-1, :].contiguous()         # (B, L-1, D)
    mask   = (target != 0)                     # PAD_ID=0

    if mask.sum() == 0:
        return H.new_zeros((), dtype=torch.float32)

    # 2) float32로 상향 (fp16 overflow 방지)
    Hctx_f = Hctx.float()

    # Positive (정답 토큰 임베딩)
    pos_vec = item_emb(target.clamp_min(0)).float()              # (B, L-1, D)
    s_pos   = (Hctx_f * pos_vec).sum(dim=2)                      # (B, L-1)

    # Negatives: uniform-sampled K개
    with torch.no_grad():
        uni_negs = torch.randint(low=2, high=item_emb.num_embeddings,  # SEQ_BASE=2
                                 size=(B, L-1, neg_k), device=H.device)
    neg_vec = item_emb(uni_negs).float()                           # (B, L-1, K, D)
    s_neg   = torch.einsum("bld,blkd->blk", Hctx_f, neg_vec)       # (B, L-1, K)

    # 3) LogSumExp로 안정적으로 정규화
    # logits = [s_pos, s_neg...] 형태로 쌓고 마스킹은 -inf로
    s_pos_expanded = s_pos.unsqueeze(2)                            # (B, L-1, 1)
    logits_all = torch.cat([s_pos_expanded, s_neg], dim=2).float() # (B, L-1, 1+K)

    # 마스크가 False인 위치는 -inf
    neg_inf = torch.finfo(logits_all.dtype).min
    logits_all = logits_all.masked_fill(~mask.unsqueeze(2), neg_inf)
    s_pos = s_pos.masked_fill(~mask, neg_inf)

    # InfoNCE: -s_pos + logsumexp([s_pos, s_neg])
    lse = torch.logsumexp(logits_all, dim=2)                       # (B, L-1)
    loss_t = (-s_pos + lse)                                        # (B, L-1)
    loss = loss_t.masked_select(mask).mean()                       # 평균

    return loss

# ================= Data & Loaders =================
ds_tr=CTRDataset(tr, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, args.segment_cols, label_col=args.label)
ds_va=CTRDataset(va, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, args.segment_cols, label_col=args.label)
ds_te=CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, args.segment_cols, label_col=None)

n_pos=float((tr[args.label]==1).sum()); n_neg=float(len(tr)-n_pos)
pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[2/9] Train label: pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

dl_tr=DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_va=DataLoader(ds_va, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_te=DataLoader(ds_te, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)

# ================= Train =================
print("[3/9] Build model ...")
bst_cfg={"nhead":args.bst_nhead, "num_layers":args.bst_layers, "dim_ff":args.bst_ffn}
num_segments=len(seg2idx)
model=DCN_DUSIN_Full(
    cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
    num_segments=num_segments, seg_oov_index=SEG_OOV,
    target_name=target_name, seq_emb_dim=args.seq_emb_dim, seq_backbone=args.seq_backbone,
    bst_cfg=bst_cfg, deep_units=args.deep_units,
    cross_layers=args.cross_layers, cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
    dropout=args.dropout, seg_momentum=args.seg_ema, use_seg_aux=args.use_seg_aux
).to(device)

criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
seg_ce = nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler=GradScaler(enabled=(device.type=="cuda"))

best_auc=-1.0; best_state=None; wait=0
final_epoch=0; final_prauc=0.0

print(f"[4/9] Train ... (backbone={args.seq_backbone}, segments={num_segments}+OOV)")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, seg_id, y in dl_tr:
        xc, seq, seg_id, y = xc.to(device), seq.to(device), seg_id.to(device), y.to(device)
        cats_dev={k:v.to(device) for k,v in cats.items()}

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits, seg_logits, H = model(xc, cats_dev, seq, seg_id)
            main_loss = criterion(logits, y)

            # seg aux
            loss_seg = 0.0
            if args.use_seg_aux and seg_logits is not None:
                loss_seg = seg_ce(seg_logits, seg_id.clamp_max(seg_logits.size(1)-1))

            # next-action aux
            loss_next = 0.0
            if args.use_next_aux:
                loss_next = next_action_aux_loss(H, seq, model.item_emb, neg_k=args.next_neg_k)

            loss = main_loss + args.seg_aux_lambda*loss_seg + args.next_aux_lambda*loss_next

        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)

    tr_loss/=len(ds_tr); scheduler.step()

    # ----- Validation -----
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, seg_id, y in dl_va:
            xc, seq, seg_id = xc.to(device), seq.to(device), seg_id.to(device)
            cats_dev={k:v.to(device) for k,v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                logits, _, _ = model(xc, cats_dev, seq, seg_id)
                loss=criterion(logits, y.to(device))
                prob=torch.sigmoid(logits)
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            va_loss += loss.item()*len(y)
            ys.append(y.numpy()); ps.append(p)
    va_loss/=len(ds_va)
    y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
    auc=roc_auc_score(y_true, y_prob); prauc=average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc, final_prauc, final_epoch = auc, prauc, epoch
        best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait+=1
        if wait>=args.patience:
            print(f"Early stopping. Best AUC={best_auc:.6f} at epoch {final_epoch}."); break

if best_state is not None:
    model.load_state_dict(best_state)

# ================= Inference =================
print("[5/9] Predict test ...")
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq, seg_id in dl_te:
        xc, seq, seg_id = xc.to(device), seq.to(device), seg_id.to(device)
        cats_dev={k:v.to(device) for k,v in cats.items()}
        with autocast(enabled=(device.type=="cuda")):
            logits, _, _ = model(xc, cats_dev, seq, seg_id)
            prob=torch.sigmoid(logits)
        p = prob.detach().cpu().numpy().astype(np.float64)
        p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
        probs.append(p)
probs=np.concatenate(probs)

# ================= Save =================
print("[6/9] Save outputs ...]")
sub_path=f"./dcnv2_{args.seq_backbone}_dusin_full_submit.csv"
meta_path=f"./dcnv2_{args.seq_backbone}_dusin_full_meta.json"
pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs}).to_csv(sub_path, index=False)
meta={
    "columns":{"continuous":cont_cols, "categorical":cand_cats, "segment_cols":args.segment_cols},
    "seq_vocab":{"type":"hash","buckets":args.HASH_BUCKETS,"pad_id":0,"oov_id":1},
    "hyperparameters":{
        "sample_subset":args.sample_subset,"max_seq_len":args.max_seq_len,
        "seq_emb_dim":args.seq_emb_dim,"dropout":args.dropout,
        "epochs":args.epochs,"batch_size":args.batch_size,"lr":args.lr,
        "cross_layers":args.cross_layers,"cross_low_rank":args.cross_low_rank,"cross_num_experts":args.cross_num_experts,
        "deep_units":args.deep_units, "seq_backbone":args.seq_backbone,
        "bst_layers":args.bst_layers,"bst_nhead":args.bst_nhead,"bst_ffn":args.bst_ffn,
        "seg_ema":args.seg_ema,"seg_aux_lambda":args.seg_aux_lambda,
        "next_aux_lambda":args.next_aux_lambda,"next_neg_k":args.next_neg_k
    },
    "performance":{"best_epoch":int(final_epoch),"AUC":float(best_auc),"PR_AUC":float(final_prauc)},
    "target_name": target_name,
    "segments":{"count":int(num_segments), "oov_index":int(SEG_OOV)}
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
print(f"[7/9] Done. \n - {sub_path}\n - {meta_path}")


Using CUDA
[1/9] Load parquet ...
[1.1] target_name=l_feat_14  cont=85  cat=32
[2/9] Train label: pos=173552 neg=8925000  pos_weight=51.426
[3/9] Build model ...
[4/9] Train ... (backbone=dien, segments=17+OOV)
Epoch 01 | Train 1.43425 | Val nan | AUC 0.732243 | PR-AUC 0.067570
Epoch 02 | Train 1.38693 | Val nan | AUC 0.735469 | PR-AUC 0.073631
Epoch 03 | Train 1.36402 | Val nan | AUC 0.741699 | PR-AUC 0.078040
[5/9] Predict test ...
[6/9] Save outputs ...]
[7/9] Done. 
 - ./dcnv2_dien_dusin_full_submit.csv
 - ./dcnv2_dien_dusin_full_meta.json


In [1]:
# -*- coding: utf-8 -*-
"""
CTR — DCN-V2 + (DIEN/BST) + DUSIN + Momentum Encoder + Multi-View Segment Prototypes + Contrastive
[Notebook, OOM-safe, NaN-safe]

핵심 추가
- Momentum(teacher) sequence encoder (MoCo-style)
- Multi-level segment prototype banks (EMA update + L2 normalize)
- Instance→Prototype contrastive loss (InfoNCE, temperature τ)
- DUSIN: segment interest activation + personal interest
- 선택: DIEN(GRU) / BST(Transformer), Next-action auxiliary loss

주의
- 실제 대용량 학습 시 batch/num_workers/precision 등을 환경에 맞춰 조정하세요.
- contrastive와 prototype bank는 학습동안 지속적으로 업데이트됩니다.
"""

import os, re, json, random, gc, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# ===================== Utils =====================
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]; toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s); toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: pass
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

def l2_normalize(x, eps=1e-12):
    return x / (x.norm(p=2, dim=-1, keepdim=True).clamp_min(eps))

# ===================== Config =====================
class Args:
    train="./train.parquet"; test="./test.parquet"
    label="clicked"; seq_col="seq"; id_col="ID"

    sample_subset=1.0
    max_seq_len=50
    HASH_BUCKETS=262_144
    seq_emb_dim=64
    dropout=0.2

    # Backbone: "dien" or "bst"
    seq_backbone="dien"
    bst_layers=2; bst_nhead=4; bst_ffn=128

    # DCN-V2
    cross_layers=3; cross_low_rank=32; cross_num_experts=4
    deep_units=[512,256,128]

    # --- DUSIN + Segments ---
    # 다층 세그먼트 뷰 정의: 아래 순서대로 레벨 0,1,2...
    segment_levels=[["gender"], ["age_group"], ["gender","age_group"]]

    # --- Contrastive ---
    use_contrastive=True
    contr_lambda=0.1         # 메인손실에 더해지는 가중치
    contr_temp=0.07          # softmax temperature
    proto_momentum=0.9       # 프로토타입 EMA 업데이트 계수
    momenc_m=0.999           # momentum encoder 계수

    # Next-action auxiliary (옵션)
    use_next_aux=False
    next_aux_lambda=0.2
    next_neg_k=32

    # Train
    epochs=3; batch_size=512; lr=1e-3; patience=2
    seed=42; device="auto"; num_workers=0; pin_memory=False
args=Args()
set_seed(args.seed); device=infer_device(args.device)

# ===================== Dummy data (if missing) =====================
def create_dummy(num_rows, is_test=False):
    rng=np.random.default_rng(0)
    df=pd.DataFrame({
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0,7,num_rows),
        "hour": rng.integers(0,24,num_rows),
        "l_feat_14": rng.integers(1000,1050,num_rows),
    })
    seqs=[]
    for _ in range(num_rows):
        fmt=rng.choice(["pipe","comma","json","empty","single"])
        L=int(rng.integers(1,args.max_seq_len+10))
        items=[str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe": seqs.append("|".join(items))
        elif fmt=="comma": seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty": seqs.append(np.nan)
        else: seqs.append(items[0])
    df[args.seq_col]=seqs
    if is_test: df[args.id_col]=np.arange(num_rows)
    else: df[args.label]=rng.integers(0,2,num_rows)
    return df

if not os.path.exists(args.train):
    print("train.parquet 없음 → 더미 20k 생성"); create_dummy(20_000,False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 더미 5k 생성"); create_dummy(5_000,True).to_parquet(args.test)

# ===================== Load & Plan =====================
print("[1/9] Load ...")
train=pd.read_parquet(args.train); test=pd.read_parquet(args.test)
assert args.label in train.columns and args.seq_col in train.columns

if 0<args.sample_subset<1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train=train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

must_drop={args.label,args.seq_col,args.id_col}
base_cols=[c for c in train.columns if c not in must_drop]
fixed_cats=[c for c in ["gender","age_group","inventory_id","day_of_week","hour"] if c in base_cols]
all_lfeats=[c for c in base_cols if c.startswith("l_feat_")]

preferred_targets=["ad_id","creative_id","l_feat_14"]
present_targets=[c for c in preferred_targets if c in base_cols]
target_name=present_targets[0] if present_targets else ("l_feat_14" if "l_feat_14" in all_lfeats else None)

CAT_CARD_MAX=200_000
cand_cats=fixed_cats.copy()
tr_tmp, va_tmp = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
for c in all_lfeats:
    nunq = max(tr_tmp[c].nunique(dropna=True), va_tmp[c].nunique(dropna=True))
    if nunq <= CAT_CARD_MAX: cand_cats.append(c)
if target_name and target_name not in cand_cats and target_name in base_cols:
    cand_cats.append(target_name)
cont_cols=[c for c in base_cols if c not in cand_cats]
print(f"[1.1] target={target_name}  cont={len(cont_cols)}  cat={len(cand_cats)}")

# ===== 세그먼트 인덱스(레벨별) =====
def build_segment_key(df, cols):
    if len(cols)==0: return pd.Series(["__ALL__"]*len(df))
    return df[cols].astype(str).agg("|".join, axis=1)

def build_multi_level_segments(df, levels):
    seg_maps=[]; seg_cards=[]; seg_ids=[]
    for cols in levels:
        key=build_segment_key(df, cols)
        cats=pd.Categorical(key)
        m={v:i for i,v in enumerate(cats.categories)}
        seg_maps.append(m)
        seg_cards.append(len(m))
        seg_ids.append(key.map(m).fillna(len(m)).astype(np.int64).values) # OOV를 마지막 인덱스
    return seg_maps, seg_cards, seg_ids

seg_maps_tmp, seg_cards_tmp, _ = build_multi_level_segments(pd.concat([tr_tmp, va_tmp]), args.segment_levels)
SEG_OOV_LIST=[card for card in seg_cards_tmp]  # 각 레벨의 OOV index == 카드inality (마지막)
del tr_tmp, va_tmp; gc.collect()

# 최종 split
tr,va=train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label]); del train; gc.collect()

# 범주형 맵
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats=pd.Categorical(tr[c]); cat_maps[c]={v:i for i,v in enumerate(cats.categories)}; cat_cards[c]=len(cats.categories)

# 시퀀스 해시 임베딩
PAD_ID, OOV_ID, SEQ_BASE = 0, 1, 2
seq_vocab_size=args.HASH_BUCKETS+SEQ_BASE
def seq_to_ids_hash(lst, max_len=50):
    ids=[SEQ_BASE+(int(t)%args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids)<max_len: ids=[PAD_ID]*(max_len-len(ids))+ids
    return np.array(ids, dtype=np.int32)

# ===================== Dataset =====================
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, segment_levels, seg_maps_list, label_col=None):
        self.df=df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len=max_seq_len; self.has_label=label_col is not None; self.label_col=label_col
        self.levels=segment_levels; self.seg_maps=seg_maps_list

        self.Xc=self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats={c:self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values for c in self.cat_cols}
        if self.has_label: self.y=self.df[self.label_col].astype(np.float32).values

        # 레벨별 segment id
        self.seg_ids=[]
        for cols, m in zip(self.levels, self.seg_maps):
            key=build_segment_key(self.df, cols)
            self.seg_ids.append(key.map(m).fillna(len(m)).astype(np.int64).values)

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        xc=torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats={c: torch.tensor(self.Xcats[c][idx], dtype=torch.long) for c in self.cat_cols}
        seq=torch.from_numpy(seq_to_ids_hash(parse_seq_string(self.df.at[idx, self.seq_col]), self.max_seq_len)).long()
        segs=[torch.tensor(sids[idx], dtype=torch.long) for sids in self.seg_ids]
        if self.has_label: return xc, cats, seq, segs, torch.tensor(self.y[idx], dtype=torch.float32)
        return xc, cats, seq, segs

# ===================== Modules =====================
class CrossLayerMix(nn.Module):
    def __init__(self, dim, low_rank=32, num_experts=4):
        super().__init__()
        self.U=nn.Parameter(torch.randn(num_experts, dim, low_rank)*0.01)
        self.V=nn.Parameter(torch.randn(num_experts, low_rank, dim)*0.01)
        self.C=nn.Parameter(torch.zeros(num_experts, dim))
        self.gating=nn.Linear(dim, num_experts, bias=False)
        self.bias=nn.Parameter(torch.zeros(dim))
    def forward(self, x0, x):
        gate=torch.softmax(self.gating(x), dim=-1)
        Xu=torch.einsum("bd,edr->ber", x, self.U)
        Xuv=torch.einsum("ber,erd->bed", Xu, self.V)+self.C
        mix=torch.einsum("be,bed->bd", gate, Xuv)
        return x0*mix + x + self.bias

class CrossNetMix(nn.Module):
    def __init__(self, dim, num_layers=3, low_rank=32, num_experts=4):
        super().__init__()
        self.layers=nn.ModuleList([CrossLayerMix(dim, low_rank, num_experts) for _ in range(num_layers)])
    def forward(self, x):
        x0=x; xl=x
        for l in self.layers: xl=l(x0, xl)
        return xl

class DINActivationUnit(nn.Module):
    def __init__(self, dim_q, dim_k, hidden=[64,32], dropout=0.0):
        super().__init__()
        in_dim=dim_q+dim_k+dim_q+dim_k
        layers=[]
        for h in hidden:
            layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]; in_dim=h
        layers += [nn.Linear(in_dim,1)]
        self.net=nn.Sequential(*layers)
    def forward(self, q, K):
        B,L,Dk = K.shape
        q_exp = q.unsqueeze(1).expand(-1,L,-1)
        z=torch.cat([q_exp, K, q_exp-K, q_exp*K], dim=2)
        return self.net(z).squeeze(2)

class DIENBackbone(nn.Module):
    def __init__(self, vocab_size, emb_dim, padding_idx=0, dropout=0.2):
        super().__init__()
        self.emb=nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.gru=nn.GRU(emb_dim, emb_dim, batch_first=True)
        self.drop=nn.Dropout(dropout)
    def forward(self, seq_ids):
        K0=self.emb(seq_ids); H,_=self.gru(K0)
        return self.drop(H)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe=torch.zeros(max_len, d_model)
        pos=torch.arange(0,max_len).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2)*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:,:x.size(1),:]

class BSTBackbone(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead=4, num_layers=2, dim_ff=128, dropout=0.2, padding_idx=0):
        super().__init__()
        self.emb=nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.pos=PositionalEncoding(emb_dim, 2048)
        enc_layer=nn.TransformerEncoderLayer(d_model=emb_dim,nhead=nhead,dim_feedforward=dim_ff,
                                             batch_first=True,dropout=dropout,activation="relu",norm_first=True)
        self.enc=nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.drop=nn.Dropout(dropout)
    def forward(self, seq_ids):
        K0=self.emb(seq_ids); K0=self.pos(K0)
        H=self.enc(K0, src_key_padding_mask=(seq_ids==PAD_ID))
        return self.drop(H)

# --------- Momentum Encoder Wrapper ---------
def momentum_update(model_q, model_k, m=0.999):
    with torch.no_grad():
        for p_q, p_k in zip(model_q.parameters(), model_k.parameters()):
            p_k.data.mul_(m).add_(p_q.data, alpha=1.0-m)

# --------- Multi-level Prototype Banks ---------
class MultiLevelPrototypeBank(nn.Module):
    """
    레벨마다 (num_segments_l + 1, D) 프로토타입 버퍼를 보유. 업데이트는 EMA + L2 Norm.
    """
    def __init__(self, num_segments_list, dim, oov_index_list, momentum=0.9):
        super().__init__()
        self.dim=dim; self.m=momentum
        self.banks=nn.ParameterList()  # not trainable but Parameter to move with .to(device)
        self.oov=oov_index_list
        for n in num_segments_list:
            bank=nn.Parameter(torch.zeros(n+1, dim), requires_grad=False)
            self.banks.append(bank)

    @torch.no_grad()
    def update_with_batch(self, level, seg_ids, reps):
        # reps:(B,D), seg_ids:(B,)
        bank=self.banks[level]
        for s in torch.unique(seg_ids):
            mask=(seg_ids==s).unsqueeze(1)
            v = reps.masked_fill(~mask, 0.0).sum(dim=0) / mask.sum().clamp_min(1.0)  # mean
            v = l2_normalize(v.unsqueeze(0)).squeeze(0)
            bank[s] = self.m*bank[s] + (1-self.m)*v
            bank[s] = l2_normalize(bank[s].unsqueeze(0)).squeeze(0)

    def lookup(self, level, seg_ids):
        bank=self.banks[level]
        v=bank[seg_ids]  # (B,D)
        return l2_normalize(v)

    def negatives(self, level, exclude_ids=None):
        # 전체 프로토타입들(정규화됨)
        bank=l2_normalize(self.banks[level])
        if exclude_ids is None:
            return bank
        # 배치에서 쓰이는 seg id는 제외 가능(선택)
        mask=torch.ones(bank.size(0), dtype=torch.bool, device=bank.device)
        mask[exclude_ids.unique()] = False
        mask = mask & torch.isfinite(bank.norm(dim=1))  # 안 쓰인 프로토타입이 0일 수 있으니 필터
        return bank[mask]

# ===================== Model =====================
class DCN_DUSIN_MOCO_Model(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, target_name, segment_levels,
                 seg_num_list, seg_oov_list, seq_emb_dim=64, seq_backbone="dien",
                 bst_cfg=None, deep_units=[512,256,128],
                 cross_layers=3, cross_low_rank=32, cross_num_experts=4,
                 dropout=0.2, proto_momentum=0.9, momenc_m=0.999):
        super().__init__()
        self.has_cont=cont_dim>0
        if self.has_cont: self.bn=nn.BatchNorm1d(cont_dim)

        # cat embeddings
        self.cat_embs=nn.ModuleDict()
        for n, card in cat_cards.items():
            dim=emb_dim_from_card(card+1)
            self.cat_embs[n]=nn.Embedding(card+1+1, dim)
        self.cat_total_dim = sum(e.embedding_dim for e in self.cat_embs.values())

        self.tab_dim = (cont_dim if self.has_cont else 0) + self.cat_total_dim

        self.cross=CrossNetMix(self.tab_dim, cross_layers, cross_low_rank, cross_num_experts)
        deep=[]; in_dim=self.tab_dim
        for h in deep_units: deep += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]; in_dim=h
        self.deep=nn.Sequential(*deep)

        self.target_name = target_name if (target_name in self.cat_embs) else None
        self.tproj=None
        if self.target_name is not None:
            tdim=self.cat_embs[self.target_name].embedding_dim
            if tdim != seq_emb_dim: self.tproj=nn.Linear(tdim, seq_emb_dim, bias=False)

        # online backbone
        if seq_backbone=="bst":
            cfg=bst_cfg or {"nhead":4,"num_layers":2,"dim_ff":128}
            self.backbone_q=BSTBackbone(seq_vocab_size, seq_emb_dim, nhead=cfg["nhead"], num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"], dropout=dropout, padding_idx=PAD_ID)
            self.backbone_k=BSTBackbone(seq_vocab_size, seq_emb_dim, nhead=cfg["nhead"], num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"], dropout=dropout, padding_idx=PAD_ID)
        else:
            self.backbone_q=DIENBackbone(seq_vocab_size, seq_emb_dim, PAD_ID, dropout)
            self.backbone_k=DIENBackbone(seq_vocab_size, seq_emb_dim, PAD_ID, dropout)

        # init momentum encoder weights
        for p_q, p_k in zip(self.backbone_q.parameters(), self.backbone_k.parameters()):
            p_k.data.copy_(p_q.data); p_k.requires_grad=False
        self.momenc_m = momenc_m

        # DUSIN act for segment attention (query = segment proto)
        self.seg_act = DINActivationUnit(seq_emb_dim, seq_emb_dim, hidden=[64,32], dropout=dropout)

        # multi-level prototype bank
        self.proto_bank = MultiLevelPrototypeBank(seg_num_list, dim=seq_emb_dim, oov_index_list=seg_oov_list, momentum=proto_momentum)
        self.levels = segment_levels

        # head: [x_cross, x_deep, v_last, vU_A, vU_S, S_i]  (same style)
        final_in = self.tab_dim + deep_units[-1] + seq_emb_dim*4
        self.head = nn.Sequential(nn.Linear(final_in,256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256,1))

        # item embedding for aux (optional)
        self.item_emb = nn.Embedding(seq_vocab_size, seq_emb_dim, padding_idx=PAD_ID)

    def personal_last(self, H, seq_ids):
        # robust "last-ish": 최근 시점에 가중치 큰 max
        B,L,D = H.shape
        t = (torch.arange(L, device=H.device).float()+1.0)[None,:,None]  # (1,L,1)
        weighted = H*t
        mask=(seq_ids!=PAD_ID).float().unsqueeze(-1)
        weighted = weighted.masked_fill(mask==0, -1e9 if H.dtype==torch.float32 else torch.finfo(H.dtype).min)
        v_last = weighted.max(dim=1).values
        v_last = torch.nan_to_num(v_last, nan=0.0, posinf=0.0, neginf=0.0)
        return v_last

    def forward(self, xc, xcats, seq_ids, seg_ids_list, update_momentum=False):
        # tabular
        outs=[]
        if self.has_cont: outs.append(self.bn(xc))
        if len(self.cat_embs)>0: outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))
        x0 = torch.cat(outs, dim=1) if len(outs)>1 else outs[0]
        x_cross = self.cross(x0); x_deep = self.deep(x0)

        # momentum update
        if self.training and update_momentum:
            momentum_update(self.backbone_q, self.backbone_k, self.momenc_m)

        # sequence enc (online q & teacher k)
        H_q = self.backbone_q(seq_ids)            # (B,L,D)
        with torch.no_grad():
            H_k = self.backbone_k(seq_ids)        # (B,L,D) — teacher
        v_last_q = self.personal_last(H_q, seq_ids)    # (B,D)
        v_last_k = self.personal_last(H_k, seq_ids).detach()

        # target emb vA
        if self.target_name is not None:
            vA = self.cat_embs[self.target_name](xcats[self.target_name])
            if self.tproj is not None: vA = self.tproj(vA)
        else:
            vA = torch.zeros_like(v_last_q)

        # 레벨별 DUSIN: proto lookup/ update / activation
        vU_S_list=[]; S_list=[]; vU_A_list=[]
        for level, seg_ids in enumerate(seg_ids_list):
            with torch.no_grad():
                self.proto_bank.update_with_batch(level, seg_ids, v_last_k)  # teacher rep로 업데이트 (안정화)

            S_i = self.proto_bank.lookup(level, seg_ids)                     # (B,D), L2-normalized
            logits_w = self.seg_act(S_i, H_q)                                # (B,L)
            neg_inf = torch.finfo(H_q.dtype).min
            logits_w = logits_w.masked_fill((seq_ids==PAD_ID), neg_inf)
            valid = (seq_ids!=PAD_ID).any(dim=1, keepdim=True)
            maxv = torch.where(valid, logits_w.max(dim=1, keepdim=True).values, torch.zeros_like(logits_w[:,:1]))
            logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
            alpha = torch.where(valid, torch.softmax(logits_w,dim=1), torch.zeros_like(logits_w))
            alpha = torch.nan_to_num(alpha, nan=0.0)
            vU_S = torch.sum(H_q * alpha.unsqueeze(2), dim=1)                # (B,D)

            vU_A = vA * S_i

            vU_S_list.append(vU_S)
            S_list.append(S_i)
            vU_A_list.append(vU_A)

        # concat (여기선 모든 레벨을 평균으로 요약해 결합 — 단순화)
        S_cat   = torch.stack(S_list, dim=0).mean(dim=0)
        vUS_cat = torch.stack(vU_S_list, dim=0).mean(dim=0)
        vUA_cat = torch.stack(vU_A_list, dim=0).mean(dim=0)

        z = torch.cat([x_cross, x_deep, v_last_q, vUA_cat, vUS_cat, S_cat], dim=1)
        logits = self.head(z).squeeze(1)
        return logits, v_last_q.detach(), v_last_k.detach(), S_list  # contrast에서 사용

# ===================== Losses =====================
def bce_logits_loss(logits, y, pos_weight):
    return nn.functional.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

def contrastive_proto_loss(q_rep, seg_ids_list, proto_bank, temp=0.07):
    """
    q_rep: (B,D) — online 개인 관심(또는 vUS) 벡터 (이미 L2 norm 아님)
    각 레벨에서: pos = 해당 세그먼트 프로토타입, neg = 동일 레벨의 모든 다른 프로토타입
    """
    q = l2_normalize(q_rep)
    total_loss=0.0; levels=len(seg_ids_list)
    for level, seg_ids in enumerate(seg_ids_list):
        # pos
        pos = proto_bank.lookup(level, seg_ids)          # (B,D) normalized
        # negs
        all_proto = proto_bank.banks[level]              # (N+1,D)
        all_proto = l2_normalize(all_proto)
        # (선택) 자기 세그먼트 제외
        # logits = q·[pos, all_proto_except_pos]^T
        pos_logit = (q*pos).sum(dim=1, keepdim=True)     # (B,1)
        # 샘플링식으로 속도절약: 전체에서 무작위 K개 뽑아도 됨(여기선 전부 사용 가능)
        # gather negatives
        # 벌크 행렬곱으로 구현
        logits_all = q @ all_proto.T                     # (B, Ntot)
        # 각 행에서 자신의 세그먼트 id index 위치 추출해서 pos와 align하려면 마스크 필요
        # 간단히 [pos, all]을 concat 후 pos index는 0으로 두고 cross-entropy
        logits = torch.cat([pos_logit, logits_all], dim=1) / temp
        labels = torch.zeros(q.size(0), dtype=torch.long, device=q.device)   # 0 이 양성
        loss = F.cross_entropy(logits, labels)
        total_loss += loss
    return total_loss / max(1, levels)

def next_action_aux_loss(H, seq_ids, item_emb, neg_k=32):
    """
    InfoNCE 스타일 다음 행동 보조손실 (fp16-safe: float32 연산)
    """
    B,L,D = H.shape
    if L<=1: return H.new_zeros((), dtype=torch.float32)
    target = seq_ids[:,1:].contiguous()
    Hctx   = H[:,:-1,:].contiguous()
    mask   = (target!=PAD_ID)
    if mask.sum()==0: return H.new_zeros((), dtype=torch.float32)
    Hctx_f = Hctx.float()
    pos_vec = item_emb(target.clamp_min(0)).float()
    s_pos = (Hctx_f*pos_vec).sum(dim=2)
    with torch.no_grad():
        uni_negs = torch.randint(low=SEQ_BASE, high=item_emb.num_embeddings, size=(B,L-1,neg_k), device=H.device)
    s_neg = torch.einsum("bld,blkd->blk", Hctx_f, item_emb(uni_negs).float())
    s_pos_e = s_pos.unsqueeze(2)
    logits_all = torch.cat([s_pos_e, s_neg], dim=2).float()
    neg_inf = torch.finfo(logits_all.dtype).min
    logits_all = logits_all.masked_fill(~mask.unsqueeze(2), neg_inf)
    s_pos = s_pos.masked_fill(~mask, neg_inf)
    lse = torch.logsumexp(logits_all, dim=2)
    loss_t = (-s_pos + lse)
    return loss_t.masked_select(mask).mean()

# ===================== Build Data =====================
ds_tr=CTRDataset(tr, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len,
                 args.segment_levels, seg_maps_tmp, label_col=args.label)
ds_va=CTRDataset(va, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len,
                 args.segment_levels, seg_maps_tmp, label_col=args.label)
ds_te=CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len,
                 args.segment_levels, seg_maps_tmp, label_col=None)

n_pos=float((tr[args.label]==1).sum()); n_neg=float(len(tr)-n_pos)
pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[2/9] Labels pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

dl_tr=DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_va=DataLoader(ds_va, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_te=DataLoader(ds_te, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=args.pin_memory)

# ===================== Build Model =====================
print("[3/9] Build model ...")
seg_num_list=[len(m) for m in seg_maps_tmp]
model=DCN_DUSIN_MOCO_Model(
    cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
    target_name=target_name, segment_levels=args.segment_levels,
    seg_num_list=seg_num_list, seg_oov_list=SEG_OOV_LIST,
    seq_emb_dim=args.seq_emb_dim, seq_backbone=args.seq_backbone,
    bst_cfg={"nhead":args.bst_nhead,"num_layers":args.bst_layers,"dim_ff":args.bst_ffn},
    deep_units=args.deep_units, cross_layers=args.cross_layers,
    cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
    dropout=args.dropout, proto_momentum=args.proto_momentum, momenc_m=args.momenc_m
).to(device)

criterion = lambda logits, y: bce_logits_loss(logits, y, pos_weight)
optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler=GradScaler(enabled=(device.type=="cuda"))

# ===================== Train =====================
best_auc=-1.0; best_state=None; wait=0
print(f"[4/9] Train ... (backbone={args.seq_backbone}, views={len(args.segment_levels)}, contrastive={args.use_contrastive})")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, segs, y in dl_tr:
        xc, seq, y = xc.to(device), seq.to(device), y.to(device)
        cats_dev={k:v.to(device) for k,v in cats.items()}
        segs_dev=[s.to(device) for s in segs]

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits, v_last_q, v_last_k, S_list = model(xc, cats_dev, seq, segs_dev, update_momentum=True)
            main_loss = criterion(logits, y)

            loss_contr = 0.0
            if args.use_contrastive:
                # contrast query로 online 개인 관심벡터 사용 (정규화는 내부 처리)
                loss_contr = contrastive_proto_loss(v_last_q, segs_dev, model.proto_bank, temp=args.contr_temp)

            loss_next = 0.0
            if args.use_next_aux:
                # teacher H를 쓰면 더 안정적이나, 여기선 online backbone의 H_q가 필요 → 모델 내부 반환을 단순화 위해 생략
                pass  # 필요시 backbone_q(seq) 결과를 저장/반환하도록 모델 수정
            loss = main_loss + args.contr_lambda*loss_contr + args.next_aux_lambda*loss_next

        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)
    tr_loss /= len(ds_tr); scheduler.step()

    # Validation
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, segs, y in dl_va:
            xc, seq = xc.to(device), seq.to(device)
            cats_dev={k:v.to(device) for k,v in cats.items()}
            segs_dev=[s.to(device) for s in segs]
            with autocast(enabled=(device.type=="cuda")):
                logits, *_ = model(xc, cats_dev, seq, segs_dev, update_momentum=False)
                loss = criterion(logits, y.to(device))
                prob = torch.sigmoid(logits)
            va_loss += loss.item()*len(y)
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            ys.append(y.numpy()); ps.append(p)
    va_loss/=len(ds_va)
    y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
    auc=roc_auc_score(y_true, y_prob); prauc=average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc=auc; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait+=1
        if wait>=args.patience:
            print(f"Early stop. Best AUC={best_auc:.6f}"); break

if best_state: model.load_state_dict(best_state)

# ===================== Inference =====================
print("[5/9] Predict ...")
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq, segs in dl_te:
        xc, seq = xc.to(device), seq.to(device)
        cats_dev={k:v.to(device) for k,v in cats.items()}
        segs_dev=[s.to(device) for s in segs]
        with autocast(enabled=(device.type=="cuda")):
            logits, *_ = model(xc, cats_dev, seq, segs_dev, update_momentum=False)
            prob=torch.sigmoid(logits)
        p=prob.detach().cpu().numpy().astype(np.float64)
        p=np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
        probs.append(p)
probs=np.concatenate(probs)

# ===================== Save =====================
print("[6/9] Save ...")
sub_path=f"./dcnv2_{args.seq_backbone}_dusin_moco_contr_submit.csv"
meta_path=f"./dcnv2_{args.seq_backbone}_dusin_moco_contr_meta.json"
pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs}).to_csv(sub_path, index=False)

meta={
    "columns":{"continuous":cont_cols,"categorical":cand_cats,"segment_levels":args.segment_levels},
    "seq_vocab":{"type":"hash","buckets":args.HASH_BUCKETS,"pad_id":0,"oov_id":1},
    "hparams":{
        "max_seq_len":args.max_seq_len,"seq_emb_dim":args.seq_emb_dim,"dropout":args.dropout,
        "epochs":args.epochs,"batch_size":args.batch_size,"lr":args.lr,
        "cross_layers":args.cross_layers,"cross_low_rank":args.cross_low_rank,"cross_num_experts":args.cross_num_experts,
        "deep_units":args.deep_units,"seq_backbone":args.seq_backbone,
        "bst_layers":args.bst_layers,"bst_nhead":args.bst_nhead,"bst_ffn":args.bst_ffn,
        "use_contrastive":args.use_contrastive,"contr_lambda":args.contr_lambda,"contr_temp":args.contr_temp,
        "proto_momentum":args.proto_momentum,"momenc_m":args.momenc_m,
        "use_next_aux":args.use_next_aux,"next_aux_lambda":args.next_aux_lambda,"next_neg_k":args.next_neg_k
    },
    "best_auc": float(best_auc),
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
print(f"Done:\n - {sub_path}\n - {meta_path}")


Using CUDA
[1/9] Load ...
[1.1] target=l_feat_14  cont=85  cat=32
[2/9] Labels pos=173552 neg=8925000  pos_weight=51.426
[3/9] Build model ...
[4/9] Train ... (backbone=dien, views=3, contrastive=True)
Epoch 01 | Train 1.42515 | Val nan | AUC 0.730448 | PR-AUC 0.069722
Epoch 02 | Train 1.40613 | Val nan | AUC 0.736337 | PR-AUC 0.074362
Epoch 03 | Train 1.38707 | Val nan | AUC 0.738750 | PR-AUC 0.076952
[5/9] Predict ...
[6/9] Save ...
Done:
 - ./dcnv2_dien_dusin_moco_contr_submit.csv
 - ./dcnv2_dien_dusin_moco_contr_meta.json
